# Import

In [1]:
import pandas as pd
import json

from datasets import load_dataset
from datasets import Dataset, DatasetDict
from presidio_analyzer import AnalyzerEngine, RecognizerRegistry
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_analyzer.predefined_recognizers import SpacyRecognizer

from projet_14.rgpd import run

/home/johan/projects/P14/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Get hugging face dataset

In [2]:
# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("ANR-MALADES/MediQAl", "mcqu")
df_train = ds["train"].to_pandas()
df_validation = ds["validation"].to_pandas()
df_test = ds["test"].to_pandas()

Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


# Analyse dataset

In [3]:
df_train.head()

,id,clinical_case,question,answer_a,answer_b,answer_c,answer_d,answer_e,correct_answers,task,medical_subject,question_type
0,7543,"Monsieur R. âgé de 37 ans, forestier de son mé...",Au sujet des vaccinations :,Le vaccin contre la fièvre jaune est obligatoire,Le vaccin contre la fièvre jaune peut se faire...,Le vaccin contre le choléra est un vaccin qui ...,Le vaccin contre l'hépatite B est conseillé ma...,Les vaccinations contre le tétanos et la polio...,A,QCU,Infectious Diseases,Understanding
1,10495,NaN,"Devant une tuméfaction cervicale médiane, juxt...",Kyste dermoïde,Laryngocèle,Kyste du tractus thyréoglosse,Grenouillette sublinguale,Lithiase salivaire,C,QCU,Otorhinolaryngology (ENT),Understanding
2,17342,Une jeune femme de 24 ans vous consulte pour a...,Le traitement de cette patiente fait appel à :,Chirurgie en urgence,Iode radio-actif en urgence,Iode radio-actif en préparation à la chirurgie,Administration d'antithyroïdiens,Aucun de ces traitements,D,QCU,Endocrinology and Metabolism,Reasoning
3,22883,"Un nouveau-né de sexe masculin, âgé de 3 jours...",Chez ce nouveau-né le bilan biologique à montr...,Ictère par Incompatibilité Rhésus,Ictère par déficit en G6PD,ictère infectieux,Ictère par incompatibilité ABO,Ictère Infectieux,D,QCU,Pediatrics,Understanding
4,17514,Un homme de 63 ans est hospitalisé pour des do...,Pour confirmer le diagnostic de maladie d'Addi...,Test de stimulation par la 1.24 corticotrophin...,Test de freinage par la dexaméthasone,Test de stimulation de l'activité rénine plasm...,Test de restriction hydrique,Scintigraphie surrénale à l'iodocholestérol,A,QCU,Endocrinology and Metabolism,Reasoning


In [4]:
# Nombres de lignes dans les différentes parties du dataset
print(f"Train dataset: {len(df_train)} rows")
print(f"Validation dataset: {len(df_validation)} rows")
print(f"Test dataset: {len(df_test)} rows")

Train dataset: 10113 rows
Validation dataset: 2561 rows
Test dataset: 4343 rows


In [5]:
# Print distinct values of "medical_subject" column
distinct_medical_subjects = df_train["medical_subject"].unique()
print(f"Distinct values of 'medical_subject' column: {distinct_medical_subjects}")

Distinct values of 'medical_subject' column: <ArrowStringArray>
[             'Infectious Diseases',        'Otorhinolaryngology (ENT)',
     'Endocrinology and Metabolism',                       'Pediatrics',
                     'Microbiology',                      'Orthopedics',
                     'Biochemistry',                       'Cardiology',
        'Gynecology and Obstetrics',                        'Radiology',
                      'Dermatology',                      'Pulmonology',
                       'Psychiatry',                     'Rheumatology',
          'Hepato-Gastroenterology',                       'Hematology',
                        'Neurology',                       'Immunology',
                    'Ophthalmology',                     'Parasitology',
                   'Nephro-Urology',             'Pathological Anatomy',
            'Occupational Medicine', 'Forensic Medicine and Toxicology',
                   'Rehabilitation',                       '

In [6]:
# Keep only the rows where "question_type" is "Reasoning"
# reasoning_df = df[df["question_type"] == "Reasoning"]
# print(f"Number of rows in reasoning_df: {reasoning_df.shape[0]}")

# Keep only the rows where "clinical_case" and "question" are not null, in each split
splits = {"train": df_train, "validation": df_validation, "test": df_test}
for name, df in splits.items():
    splits[name] = df.dropna(subset=["clinical_case", "question"])
    print(f"{name}: {len(df)} rows, {len(splits[name])} after dropping null values")

train: 10113 rows, 3935 after dropping null values
validation: 2561 rows, 999 after dropping null values
test: 4343 rows, 1996 after dropping null values


In [7]:
splits["train"].head()

,id,clinical_case,question,answer_a,answer_b,answer_c,answer_d,answer_e,correct_answers,task,medical_subject,question_type
0,7543,"Monsieur R. âgé de 37 ans, forestier de son mé...",Au sujet des vaccinations :,Le vaccin contre la fièvre jaune est obligatoire,Le vaccin contre la fièvre jaune peut se faire...,Le vaccin contre le choléra est un vaccin qui ...,Le vaccin contre l'hépatite B est conseillé ma...,Les vaccinations contre le tétanos et la polio...,A,QCU,Infectious Diseases,Understanding
2,17342,Une jeune femme de 24 ans vous consulte pour a...,Le traitement de cette patiente fait appel à :,Chirurgie en urgence,Iode radio-actif en urgence,Iode radio-actif en préparation à la chirurgie,Administration d'antithyroïdiens,Aucun de ces traitements,D,QCU,Endocrinology and Metabolism,Reasoning
3,22883,"Un nouveau-né de sexe masculin, âgé de 3 jours...",Chez ce nouveau-né le bilan biologique à montr...,Ictère par Incompatibilité Rhésus,Ictère par déficit en G6PD,ictère infectieux,Ictère par incompatibilité ABO,Ictère Infectieux,D,QCU,Pediatrics,Understanding
4,17514,Un homme de 63 ans est hospitalisé pour des do...,Pour confirmer le diagnostic de maladie d'Addi...,Test de stimulation par la 1.24 corticotrophin...,Test de freinage par la dexaméthasone,Test de stimulation de l'activité rénine plasm...,Test de restriction hydrique,Scintigraphie surrénale à l'iodocholestérol,A,QCU,Endocrinology and Metabolism,Reasoning
5,17623,Une femme de 61 ans vient consulter pour asthé...,Parmi les propositions suivantes concernant le...,L'effondrement de la fixation thyroïdienne suf...,Un taux plasmatique de T3 abaissé suffit à aff...,Il n'y a pas d'hypothyroïdie primitive sans él...,La prolactine peut être élevée,La FSH est basse et s'élève sous métopirone,D,QCU,Endocrinology and Metabolism,Understanding


# Build SFT JSONL file from dataset

In [8]:

SYSTEM_PROMPT = "Tu es un assistant médical expert. Réponds de façon précise et concise."


def to_messages(row) -> list[dict]:
    answers = [row[f"answer_{l.strip().lower()}"] for l in row["correct_answers"].split(",")]
    answer = answers[0] if len(answers) == 1 else "\n".join(f"- {a}" for a in answers)
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{row['clinical_case']}\n\n{row['question']}"},
        {"role": "assistant", "content": answer},
    ]


# Un fichier JSONL par split : ../data/sft_mediqal_mcqu_{train,validation,test}.jsonl
sft_ds = DatasetDict()
for name, df in splits.items():
    df = df.assign(messages=df.apply(to_messages, axis=1))
    sft_ds[name] = Dataset.from_pandas(
        df[["id", "medical_subject", "question_type", "messages"]], preserve_index=False
    )
    sft_ds[name].to_json(f"../data/sft_mediqal_mcqu_{name}.jsonl", force_ascii=False)

sft_ds["train"][0]["messages"]


Creating json from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 4/4 [00:00<00:00, 50.44ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 51.49ba/s]

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 56.69ba/s]

[{'role': 'system',
  'content': 'Tu es un assistant médical expert. Réponds de façon précise et concise.'},
 {'role': 'user',
  'content': "Monsieur R. âgé de 37 ans, forestier de son métier, doit partir travailler pendant 2 ans au Gabon dans la région de Lambaréné. Monsieur R. a l'intention de se faire accompagner de son épouse âgée de 36 ans et de son fils âgé de 5 ans. Ils logeront à quelques kilomètres de la forêt mais Mr R., de par son métier, passera la plupart de son temps au milieu des bois. Ils consultent 4 mois avant le départ pour se renseigner au sujet des vaccinations et de la chimioprophylaxie du paludisme. Vous lui expliquez les conseils d'hygiène, les éléments de prévention essentiels et les différents modes de contamination des maladies présentes dans la région\n\nAu sujet des vaccinations :"},
 {'role': 'assistant',
  'content': 'Le vaccin contre la fièvre jaune est obligatoire'}]

In [9]:
print(sft_ds)

DatasetDict({
    train: Dataset({
        features: ['id', 'medical_subject', 'question_type', 'messages'],
        num_rows: 3935
    })
    validation: Dataset({
        features: ['id', 'medical_subject', 'question_type', 'messages'],
        num_rows: 999
    })
    test: Dataset({
        features: ['id', 'medical_subject', 'question_type', 'messages'],
        num_rows: 1996
    })
})


# Anonymisation des données

In [10]:
# Anonymisation de chaque split : ../data/anonymized/sft_mediqal_mcqu_{split}.anonymized.jsonl
# Seules les questions (user) sont anonymisées : les réponses sont des options de QCM sans donnée patient,
# où Presidio prend des termes médicaux pour des noms (« Héparine IV » -> <PERSON>)
reports = {
    name: run(f"../data/sft_mediqal_mcqu_{name}.jsonl", lang="fr", output_dir="../data/anonymized", roles=["user"])
    for name in sft_ds
}
pd.DataFrame(reports).T[["records", "records_modified", "persons_replaced"]]

,records,records_modified,persons_replaced
train,3935,1315,2333
validation,999,314,564
test,1996,670,1215


In [11]:
# DataFrame des champs "content" des jsonl anonymisés : une ligne par message (hors system)
rows = []
for name in sft_ds:
    with open(f"../data/anonymized/sft_mediqal_mcqu_{name}.anonymized.jsonl", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            for message in record["messages"]:
                if message["role"] != "system":
                    rows.append({"split": name, "id": record["id"], "role": message["role"], "content": message["content"]})

contents_df = pd.DataFrame(rows)
contents_df["n_person"] = contents_df["content"].str.count("<PERSON>")
print(f"{len(contents_df)} messages, dont {(contents_df['n_person'] > 0).sum()} avec au moins une balise <PERSON>")
contents_df.head()

13860 messages, dont 2299 avec au moins une balise <PERSON>


,split,id,role,content,n_person
0,train,7543,user,"Monsieur <PERSON> âgé de 37 ans, forestier de ...",3
1,train,7543,assistant,Le vaccin contre la fièvre jaune est obligatoire,0
2,train,17342,user,Une jeune femme de 24 ans vous consulte pour a...,0
3,train,17342,assistant,Administration d'antithyroïdiens,0
4,train,22883,user,"Un nouveau-né de sexe masculin, âgé de 3 jours...",0


In [12]:
import re


def person_contexts(text: str, width: int = 60) -> list[str]:
    """Extrait autour de chaque balise <PERSON>, avec `width` caractères de chaque côté."""
    return [text[max(0, m.start() - width) : m.end() + width].replace("\n", " ") for m in re.finditer("<PERSON>", text)]


# Une ligne par balise <PERSON>, pour vérifier les remplacements
occurrences_df = contents_df[["split", "id", "role"]].copy()
occurrences_df["context"] = contents_df["content"].apply(person_contexts)
occurrences_df = occurrences_df.explode("context").dropna(subset=["context"])

pd.set_option("display.max_colwidth", None)
print(f"{len(occurrences_df)} occurrences de <PERSON>")
occurrences_df.sample(20, random_state=42)

3354 occurrences de <PERSON>


,split,id,role,context
1708,train,16715,user,"Monsieur <PERSON>..., 56 ans, professeur de lycée, de retour de voyage au Sén"
1294,train,1919,user,"Monsieur <PERSON> 80 ans, sans antécédent particulier, est hospitalisé en der"
9980,test,11314,user,"n psycho-motrice Parallèlement, vous expliquez à la mère d'<PERSON> que l'administration quotidienne de Fluor ne remplace pas l"
13340,test,6128,user,"M. <PERSON>.., 74 ans, est hospitalisé pour des douleurs inguino-crural"
10578,test,17628,user,"obe oculaire facilement dépressible. Le poids est de 48 kg. <PERSON> sanguin : Na = 165 mmol/l, K = 5,2 mmol/l, CI = 118 mmol/l,"
13628,test,17558,user,"montre une hépatomégalie. Le poids est à 86 kg pour 1,68 m. <PERSON> examens biologiques donnent : glycémie à 15 mmol/l ; hémogl"
10154,test,12823,user,"800/mm3 (90 % de polynucléaires neutrophiles), Hb 180 g/l- <PERSON> artérielle (sans apport d'oxygène) : pH 7,30, PaO2 45 mmHg,"
5432,train,10736,user,"Mme <PERSON>, 72 ans, est hospitalisée d'urgence pour une épistaxis grav"
12998,test,13793,user,"Monsieur <PERSON>.. <PERSON>, âgé de 31 ans, est hospitalisé en psychiatrie a"
2894,train,13632,user,"Mr <PERSON>, âge de 43 ans est admis en psychiatrie pour insomnie. Dès"
